In [71]:
import apache_beam as beam
import arxiv 

from apache_beam.dataframe.convert import to_dataframe
from datetime import datetime

In [72]:
query_keywords = [
    "\"image segmentation\"",
    "\"self-supervised learning\"",
    "\"representation learning\"",
    "\"image generation\"",
    "\"object detection\"",
    "\"transfer learning\"",
    "\"transformers\"",
    "\"adversarial training",
    "\"generative adversarial networks\"",
    "\"model compressions\"",
    "\"image segmentation\"",
    "\"few-shot learning\"",
    "\"natural language\"",
    "\"graph\"",
    "\"colorization\"",
    "\"depth estimation\"",
    "\"point cloud\"",
    "\"structured data\"",
    "\"optical flow\"",
    "\"reinforcement learning\"",
    "\"super resolution\"",
    "\"attention\"",
    "\"tabular\"",
    "\"unsupervised learning\"",
    "\"semi-supervised learning\"",
    "\"explainable\"",
    "\"radiance field\"",
    "\"decision tree\"",
    "\"time series\"",
    "\"molecule\"",
    "\"physics\"",
    "\"graphics\"",
    "\"ray tracing\"",
    "\"optical flow\"",
    "\"photogrametry\"",
]

In [73]:
import typing


class ArxivEntries(typing.NamedTuple):
    terms: typing.List[str]
    titles: str
    abstracts: str
    entry_ids: str
    pdf_urls: str | None


In [74]:
client = arxiv.Client(num_retries=20, page_size=100)

def query_with_keywords(query):
    search = arxiv.Search(
        query=query, max_results=20, sort_by=arxiv.SortCriterion.LastUpdatedDate,
    )

    for res in client.results(search):
        if res.primary_category in ["cs.CV", "stat.ML", "cs.LG"]:
            print(res.title)
            yield beam.Row(
                terms= res.categories,
                titles= res.title,
                abstracts= res.summary,
                entry_ids= res.entry_id,
                pdf_urls= res.pdf_url,

            )

In [75]:
%%writefile setup.py

import setuptools


NAME = "gather_arxiv_data"
VERSION = "0.1.0"
REQUIRED_PACKAGES = [
    "apache_beam==2.32.0",
    "pandas==1.3.2",
    "arxiv==1.4.2",
    "google_cloud_storage==1.42.1",
]


setuptools.setup(
    name=NAME,
    version=VERSION,
    install_requires=REQUIRED_PACKAGES,
    packages=setuptools.find_packages(),
    include_package_data=True,
)

Overwriting setup.py


In [76]:
gcs_bucket_name = "arxiv-data-nlp"
gcp_project = "42" # Specify this.

pipeline_args = {
    "job_name": f'arxiv-data-{datetime.utcnow().strftime("%y%m%d-%H%M%S")}',
    "num_workers": "4",
    "runner": "DirectRunner",
    "setup_file": "./setup.py",
    "project": gcp_project,
    "region": "us-central1",
    "temp_location": f"arxiv/temp",
#    "gcs_location": f"gs://{gcs_bucket_name}",
#   "staging_location": f"gs://{gcs_bucket_name}/staging",
    "save_main_session": "True",
}

# Convert the dictionary to a list of (argument, value) tuples and then flatten the list.
pipeline_args = [(f"--{k}", v) for k, v in pipeline_args.items()]
pipeline_args = [x for y in pipeline_args for x in y]

print("args:")
print(pipeline_args)



args:
['--job_name', 'arxiv-data-260415-222000', '--num_workers', '4', '--runner', 'DirectRunner', '--setup_file', './setup.py', '--project', '42', '--region', 'us-central1', '--temp_location', 'arxiv/temp', '--save_main_session', 'True']


C:\Users\peapo\AppData\Local\Temp\ipykernel_16960\3370402581.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "job_name": f'arxiv-data-{datetime.utcnow().strftime("%y%m%d-%H%M%S")}',


In [77]:
with beam.Pipeline(argv=pipeline_args) as pipeline:
    keywords = pipeline | beam.Create(query_keywords)
    records = keywords | beam.FlatMap(query_with_keywords).with_output_types(ArxivEntries)
    _ = to_dataframe(records).to_csv(
        f"arxiv/sample.csv", index=False,
        encoding="utf-8"
    )

Detecting and refurbishing ground truth errors during training of deep learning-based echocardiography segmentation models
Scaling In-Context Segmentation with Hierarchical Supervision
GroupKAN: Efficient Kolmogorov-Arnold Networks via Grouped Spline Modeling
DeferredSeg: A Multi-Expert Deferral Framework for Trustworthy Medical Image Segmentation
MedVeriSeg: Teaching MLLM-Based Medical Segmentation Models to Verify Query Validity Without Extra Training
On Efficient Variants of Segment Anything Model: A Survey
PR-MaGIC: Prompt Refinement Via Mask Decoder Gradient Flow For In-Context Segmentation
Efficient KernelSHAP Explanations for Patch-based 3D Medical Image Segmentation
PanoSAMic: Panoramic Image Segmentation from SAM Feature Encoding and Dual View Fusion
RADA: Region-Aware Dual-encoder Auxiliary learning for Barely-supervised Medical Image Segmentation
Data-Efficient Semantic Segmentation of 3D Point Clouds via Open-Vocabulary Image Segmentation-based Pseudo-Labeling
TAMISeg: Text

In [78]:
#!gsutil ls -R gs://{gcs_bucket_name}/arxiv/

In [79]:
#!gsutil cp gs://arxiv-data-nlp/arxiv/sample.csv-00000-of-00020 .

In [82]:
import pandas as pd


df = pd.read_csv("arxiv/sample.csv-00000-of-00001")
df.head()

,terms,titles,abstracts,entry_ids,pdf_urls
0,"['cs.CV', 'cs.AI']",Detecting and refurbishing ground truth errors...,Deep learning-based medical image segmentation...,http://arxiv.org/abs/2604.12832v1,https://arxiv.org/pdf/2604.12832v1
1,['cs.CV'],Scaling In-Context Segmentation with Hierarchi...,In-context learning (ICL) enables medical imag...,http://arxiv.org/abs/2604.12752v1,https://arxiv.org/pdf/2604.12752v1
2,['cs.CV'],GroupKAN: Efficient Kolmogorov-Arnold Networks...,Medical image segmentation demands models that...,http://arxiv.org/abs/2511.05477v2,https://arxiv.org/pdf/2511.05477v2
3,['cs.CV'],DeferredSeg: A Multi-Expert Deferral Framework...,Segmentation models based on deep neural netwo...,http://arxiv.org/abs/2604.12411v1,https://arxiv.org/pdf/2604.12411v1
4,['cs.CV'],MedVeriSeg: Teaching MLLM-Based Medical Segmen...,Despite recent advances in MLLM-based medical ...,http://arxiv.org/abs/2604.10242v2,https://arxiv.org/pdf/2604.10242v2


## Acknowledgements

* [Lukas Schwab](https://github.com/lukasschwab)
* [Robert Bradshaw](https://www.linkedin.com/in/robert-bradshaw-1b48a07/)